In [1]:
from google.colab import files
uploaded = files.upload()

Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [2]:
import pandas as pd

dataset = pd.read_csv("train.csv")
submission_set = pd.read_csv("test.csv")
print(dataset.shape)
print(submission_set.shape)

(42000, 785)
(28000, 784)


In [3]:
import random
import numpy as np
import torch
import torch.nn as nn

In [4]:
SEED = 42
def set_seed(seed):
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	if torch.cuda.is_available():
		torch.cuda.manual_seed_all(seed)
		torch.backends.cudnn.deterministic = True
		torch.backends.cudnn.benchmark = False
set_seed(SEED)

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
# Separate labels and features of the dataset
dataset_labels = dataset.iloc[:,0].values
dataset_features = dataset.iloc[:,1:].values
submission_features = submission_set.values

# Transfer Numpy Array as PyTorch Tensor
X_dataset_tensor = torch.tensor(dataset_features, dtype=torch.float32)
y_dataset_tensor = torch.tensor(dataset_labels, dtype=torch.long)
X_submission_tensor = torch.tensor(submission_features, dtype=torch.float32)

# Normalization
X_dataset_tensor = X_dataset_tensor / 255.0
X_submission_tensor = X_submission_tensor / 255.0

# Fit in CNN
X_dataset_tensor = X_dataset_tensor.view(-1, 1, 28, 28)
X_submission_tensor = X_submission_tensor.view(-1, 1, 28, 28)

In [7]:
# Split dataset_tensor into training and validation sets

from sklearn.model_selection import StratifiedKFold

five_cross_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

In [8]:
import torchvision.transforms.v2 as transforms
from torch.utils.data import DataLoader, Dataset, TensorDataset

train_transform = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomAffine(degrees=0, scale=(0.9, 1.1))
])

class AugmentedDataset(Dataset):
  def __init__(self, features, labels, transform=None):
    self.features = features
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    x = self.features[idx]
    y = self.labels[idx]

    if self.transform:
      x = self.transform(x)

    return x, y


BATCH_SIZE = 32

def make_fold_loaders(train_idx, val_idx):
  train_dataset = AugmentedDataset(
      X_dataset_tensor[train_idx],
      y_dataset_tensor[train_idx],
      transform=train_transform
  )
  val_dataset = AugmentedDataset(
      X_dataset_tensor[val_idx],
      y_dataset_tensor[val_idx]
  )

  train_dataloader = DataLoader(
      train_dataset,
      batch_size=BATCH_SIZE,
      shuffle=True
  )

  validation_dataloader = DataLoader(
      val_dataset,
      batch_size=BATCH_SIZE,
      shuffle=False
  )

  return train_dataloader, validation_dataloader

In [9]:
# Create a convolutional neural network
class TinyVGG_0(nn.Module):
  """
  Model architecture copying TinyVGG from:
  https://poloclub.github.io/cnn-explainer/
  """
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int, kernel_size: int):
    super().__init__()
    self.block_1 = nn.Sequential(
        nn.Conv2d(
            in_channels=input_shape,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=1
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=1
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
    )
    self.block_2 = nn.Sequential(
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=1
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=1
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
    )
    self.block_3 = nn.Sequential(
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=1
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=1
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Dropout(0.3),
        nn.Linear(
            in_features=hidden_units*3*3,
            out_features=output_shape
        )
    )

  def forward(self, x: torch.Tensor):
    x = self.block_1(x)
    x = self.block_2(x)
    x = self.block_3(x)
    x = self.classifier(x)
    return x

In [10]:
# Create another convolutional neural network
class TinyVGG_1(nn.Module):
  """
  Model architecture copying TinyVGG from:
  https://poloclub.github.io/cnn-explainer/
  """
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int, kernel_size: int):
    super().__init__()
    self.block_1 = nn.Sequential(
        nn.Conv2d(
            in_channels=input_shape,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=kernel_size // 2
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=kernel_size // 2
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
    )
    self.block_2 = nn.Sequential(
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=kernel_size // 2
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=kernel_size,
            stride=1,
            padding=kernel_size // 2
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Dropout(0.3),
        nn.Linear(
            in_features=hidden_units*7*7,
            out_features=output_shape
        )
    )

  def forward(self, x: torch.Tensor):
    x = self.block_1(x)
    x = self.block_2(x)
    x = self.classifier(x)
    return x

In [11]:
models = [
  lambda: TinyVGG_0(
    input_shape=1,
    hidden_units=64,
    output_shape=10,
    kernel_size=3
  ).to(device),

  lambda: TinyVGG_1(
    input_shape=1,
    hidden_units=32,
    output_shape=10,
    kernel_size=5
  ).to(device),

  lambda: TinyVGG_1(
    input_shape=1,
    hidden_units=32,
    output_shape=10,
    kernel_size=7
  ).to(device)
]




In [12]:
from timeit import default_timer as Timer

def train_step(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    accuracy_fn,
    device: torch.device = device
):
  train_loss, train_acc = 0, 0
  model.to(device)
  model.train()
  for batch, (X_train, y_train) in enumerate(train_dataloader):
    X_train, y_train = X_train.to(device), y_train.to(device)

    y_train_pred = model(X_train)
    loss = loss_fn(y_train_pred, y_train)
    acc = accuracy_fn(
        y_true=y_train,
        y_pred=y_train_pred.argmax(dim=1)
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    train_acc += acc

  train_loss /= len(train_dataloader)
  train_acc /= len(train_dataloader)
  print(f"Train loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")

def test_step(
    model: torch.nn.Module,
    test_dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    accuracy_fn,
    device: torch.device = device
):
  test_loss, test_acc = 0, 0
  model.eval()
  with torch.inference_mode():
    for X_test, y_test in test_dataloader:
      X_test, y_test = X_test.to(device), y_test.to(device)
      y_test_pred = model(X_test)
      test_loss += loss_fn(y_test_pred, y_test).item()
      test_acc += accuracy_fn(
          y_true=y_test,
          y_pred=y_test_pred.argmax(dim=1)
      )

    test_loss /= len(test_dataloader)
    test_acc /= len(test_dataloader)
    print(f"Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.5f}%")

  return test_acc


In [13]:
# Define accuracy function
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = (correct/len(y_pred)) * 100
  return acc

In [14]:
import copy
from tqdm.auto import tqdm

best_weights_dict = {}
best_acc_dict = {}

epochs = 30

for fold, (train_idx, val_idx) in enumerate(
    five_cross_cv.split(
        np.zeros(len(dataset_labels)),
        dataset_labels
    )
):

  # Create DataLoaders for the current fold
  train_dataloader, validation_dataloader = make_fold_loaders(
      train_idx,
      val_idx
  )

  for i, make_model in enumerate(models):

    # Initialize a fresh model
    model = make_model()

    # Initialize the best accuracy and the best weight
    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    # Setup loss function and optimizer
    loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
      params=model.parameters(),
      lr=0.001,
      weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
      optimizer,
      T_max=epochs,
      eta_min=1e-5
    )

    print(f"\n----------------Flod {fold + 1} | Model {i}------------------")

    for epoch in tqdm(range(epochs)):

      print(f"Epoch:{epoch}\n----------------")

      train_step(
          model=model,
          train_dataloader=train_dataloader,
          loss_fn=loss_fn,
          optimizer=optimizer,
          accuracy_fn=accuracy_fn,
          device=device
      )

      val_acc = test_step(
                model=model,
                test_dataloader=validation_dataloader,
                loss_fn=loss_fn,
                accuracy_fn=accuracy_fn,
                device=device
            )

      if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        print(f"Found new best model! Acc: {best_acc:.5f}%")

      scheduler.step()

    best_weights_dict[(fold, i)] = best_model_wts
    best_acc_dict[(fold, i)] = best_acc

    print(f"Fold {fold + 1} | Model {i} training finished."
      f"Best Validation Acc: {best_acc:.5f}%"
    )





----------------Flod 1 | Model 0------------------


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch:0
----------------
Train loss: 0.20358 | Train accuracy: 93.68%
Test loss: 0.06471 | Test accuracy: 98.22956%
Found new best model! Acc: 98.22956%
Epoch:1
----------------
Train loss: 0.08272 | Train accuracy: 97.42%
Test loss: 0.04761 | Test accuracy: 98.66920%
Found new best model! Acc: 98.66920%
Epoch:2
----------------
Train loss: 0.06967 | Train accuracy: 97.77%
Test loss: 0.03723 | Test accuracy: 99.00190%
Found new best model! Acc: 99.00190%
Epoch:3
----------------
Train loss: 0.06027 | Train accuracy: 98.09%
Test loss: 0.04510 | Test accuracy: 98.78802%
Epoch:4
----------------
Train loss: 0.05482 | Train accuracy: 98.25%
Test loss: 0.03382 | Test accuracy: 99.09696%
Found new best model! Acc: 99.09696%
Epoch:5
----------------
Train loss: 0.05134 | Train accuracy: 98.45%
Test loss: 0.03929 | Test accuracy: 98.93061%
Epoch:6
----------------
Train loss: 0.04897 | Train accuracy: 98.50%
Test loss: 0.03334 | Test accuracy: 99.15637%
Found new best model! Acc: 99.15637%
Epo

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch:0
----------------
Train loss: 0.23838 | Train accuracy: 92.40%
Test loss: 0.06419 | Test accuracy: 98.06321%
Found new best model! Acc: 98.06321%
Epoch:1
----------------
Train loss: 0.10051 | Train accuracy: 96.81%
Test loss: 0.04430 | Test accuracy: 98.70485%
Found new best model! Acc: 98.70485%
Epoch:2
----------------
Train loss: 0.08097 | Train accuracy: 97.48%
Test loss: 0.05437 | Test accuracy: 98.40779%
Epoch:3
----------------
Train loss: 0.07040 | Train accuracy: 97.82%
Test loss: 0.04088 | Test accuracy: 98.82367%
Found new best model! Acc: 98.82367%
Epoch:4
----------------
Train loss: 0.06269 | Train accuracy: 98.05%
Test loss: 0.03350 | Test accuracy: 98.90684%
Found new best model! Acc: 98.90684%
Epoch:5
----------------
Train loss: 0.05392 | Train accuracy: 98.31%
Test loss: 0.04518 | Test accuracy: 98.56226%
Epoch:6
----------------
Train loss: 0.05462 | Train accuracy: 98.30%
Test loss: 0.03399 | Test accuracy: 98.97814%
Found new best model! Acc: 98.97814%
Epo

  0%|          | 0/30 [00:00<?, ?it/s]

... (21 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch:0
----------------
Train loss: 0.24525 | Train accuracy: 92.24%
Test loss: 0.06068 | Test accuracy: 97.86122%
Found new best model! Acc: 97.86122%
Epoch:1
----------------
Train loss: 0.09755 | Train accuracy: 97.04%
Test loss: 0.04607 | Test accuracy: 98.63356%
Found new best model! Acc: 98.63356%
Epoch:2
----------------
Train loss: 0.08142 | Train accuracy: 97.51%
Test loss: 0.03937 | Test accuracy: 98.76426%
Found new best model! Acc: 98.76426%
Epoch:3
----------------
Train loss: 0.07125 | Train accuracy: 97.87%
Test loss: 0.07736 | Test accuracy: 97.75428%
Epoch:4
----------------
Train loss: 0.06245 | Train accuracy: 98.10%
Test loss: 0.03104 | Test accuracy: 99.07319%
Found new best model! Acc: 99.07319%
Epoch:5
----------------
Train loss: 0.05721 | Train accuracy: 98.14%
Test loss: 0.03778 | Test accuracy: 98.78802%
Epoch:6
----------------
Train loss: 0.05179 | Train accuracy: 98.42%
Test loss: 0.02402 | Test accuracy: 99.19202%
Found new best model! Acc: 99.19202%
Epo

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch:0
----------------
Train loss: 0.23620 | Train accuracy: 92.52%
Test loss: 0.05227 | Test accuracy: 98.24144%
Found new best model! Acc: 98.24144%
Epoch:1
----------------
Train loss: 0.09956 | Train accuracy: 96.93%
Test loss: 0.05866 | Test accuracy: 98.24144%
Epoch:2
----------------
Train loss: 0.07587 | Train accuracy: 97.67%
Test loss: 0.05733 | Test accuracy: 98.33650%
Found new best model! Acc: 98.33650%
Epoch:3
----------------
Train loss: 0.06714 | Train accuracy: 97.97%
Test loss: 0.03494 | Test accuracy: 98.87120%
Found new best model! Acc: 98.87120%
Epoch:4
----------------
Train loss: 0.06171 | Train accuracy: 98.10%
Test loss: 0.04522 | Test accuracy: 98.62167%
Epoch:5
----------------
Train loss: 0.05543 | Train accuracy: 98.29%
Test loss: 0.03109 | Test accuracy: 98.95437%
Found new best model! Acc: 98.95437%
Epoch:6
----------------
Train loss: 0.05119 | Train accuracy: 98.40%
Test loss: 0.02558 | Test accuracy: 99.01378%
Found new best model! Acc: 99.01378%
Epo

In [15]:
from re import sub
import torch.nn.functional as F

submission_dataloader = DataLoader(
    X_submission_tensor,
    batch_size=32,
    shuffle=False
)

ensemble_probs = torch.zeros(
    (len(X_submission_tensor), 10),
    device=device)

with torch.inference_mode():

  for (fold, i), weights in best_weights_dict.items():

    model = models[i]()       # Decide which kind of model architecture to use
    model.load_state_dict(weights)   # load the trained weight into that particular architecture
    model.eval()

    all_probs = []

    for X_batch in submission_dataloader:
      logits = model(X_batch.to(device))
      probs = F.softmax(logits, dim=1)
      all_probs.append(probs)

    ensemble_probs += torch.cat(all_probs, dim=0)

ensemble_probs /= len(best_weights_dict)

predictions = ensemble_probs.argmax(dim=1).cpu().numpy()

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(predictions) + 1),
    "Label": predictions
})

assert len(submission) == 28000
assert np.issubdtype(predictions.dtype, np.integer)
assert ((predictions >= 0) & (predictions <= 9)).all()

submission.to_csv("submission.csv", index=False)
print("Submission rows:", len(submission))
display(submission.head())

Submission rows: 28000


,ImageId,Label
0,1,2
1,2,0
2,3,9
3,4,0
4,5,3


In [16]:
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>